In [ ]:
"""
=============================================================
    HFT SCORE + UT BOT HYBRID BACKTEST (NSE STOCKS)
=============================================================

STRATEGY:
---------
1. Detect possible HFT / smart money activity using:
    • Volume spikes
    • Spread compression
    • ATR expansion
    • Momentum bursts
    • VWAP deviation
    • Intraday acceleration

2. Generate HFT SCORE

3. Compare:
    A) HFT strategy only
    B) UT Bot only
    C) HFT + UT Bot combined

4. Output:
    • Final Capital
    • Win Rate
    • Sharpe Ratio
    • Max Drawdown
    • Excel Trade Logs
    • Equity Curves
    • Normal Distribution Graphs

=============================================================
"""

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================

INITIAL_CAPITAL     = 100000
MAX_POSITIONS       = 8
STOP_LOSS_PCT       = 0.05
TAKE_PROFIT_PCT     = 0.12

LOOKBACK_PERIOD     = "5y"

ATR_PERIOD          = 14
UT_MULT             = 1.0

HFT_SCORE_THRESHOLD = 4.0

MAX_WORKERS         = 20

# ============================================================
# LOAD NSE TICKERS
# ============================================================

def load_nse_tickers():

    df = pd.read_csv("data/EQUITY_L.csv")

    symbols = (
        df["SYMBOL"]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )

    return [s + ".NS" for s in symbols if "&" not in s]

# ============================================================
# NORMALIZE INDEX
# ============================================================

def normalize_index(df):

    idx = df.index

    if hasattr(idx, "tz") and idx.tz is not None:
        idx = idx.tz_convert("UTC").tz_localize(None)

    df.index = idx.normalize()

    return df

# ============================================================
# UT BOT
# ============================================================

def compute_utbot(df):

    df = df.copy()

    tr = np.maximum(
        df["High"] - df["Low"],
        np.maximum(
            abs(df["High"] - df["Close"].shift()),
            abs(df["Low"] - df["Close"].shift())
        )
    )

    df["ATR"] = tr.rolling(ATR_PERIOD).mean()

    df["Upper"] = df["Close"] - (UT_MULT * df["ATR"])
    df["Lower"] = df["Close"] + (UT_MULT * df["ATR"])

    trend = [1]

    for i in range(1, len(df)):

        if df["Close"].iloc[i] > df["Lower"].iloc[i - 1]:
            trend.append(1)

        elif df["Close"].iloc[i] < df["Upper"].iloc[i - 1]:
            trend.append(-1)

        else:
            trend.append(trend[-1])

    df["Trend"] = trend

    df["UT_Buy"] = (
        (df["Trend"] == 1)
        &
        (df["Trend"].shift() == -1)
    )

    df["UT_Sell"] = (
        (df["Trend"] == -1)
        &
        (df["Trend"].shift() == 1)
    )

    return df

# ============================================================
# HFT SCORE ENGINE
# ============================================================

def compute_hft_score(df):

    df = df.copy()

    # --------------------------------------------------------
    # Volume Spike
    # --------------------------------------------------------

    df["Vol_MA20"] = df["Volume"].rolling(20).mean()

    df["Volume_Spike"] = (
        df["Volume"] / df["Vol_MA20"]
    )

    # --------------------------------------------------------
    # Spread Compression
    # --------------------------------------------------------

    df["Spread"] = (
        (df["High"] - df["Low"]) / df["Close"]
    )

    df["Spread_MA"] = df["Spread"].rolling(20).mean()

    # --------------------------------------------------------
    # Momentum Burst
    # --------------------------------------------------------

    df["Momentum_3"] = (
        df["Close"].pct_change(3)
    )

    # --------------------------------------------------------
    # ATR Expansion
    # --------------------------------------------------------

    tr = np.maximum(
        df["High"] - df["Low"],
        np.maximum(
            abs(df["High"] - df["Close"].shift()),
            abs(df["Low"] - df["Close"].shift())
        )
    )

    df["ATR"] = tr.rolling(14).mean()

    df["ATR_MA"] = df["ATR"].rolling(20).mean()

    # --------------------------------------------------------
    # VWAP Proxy
    # --------------------------------------------------------

    typical = (
        df["High"] +
        df["Low"] +
        df["Close"]
    ) / 3

    df["VWAP"] = (
        (typical * df["Volume"]).rolling(20).sum()
        /
        df["Volume"].rolling(20).sum()
    )

    df["VWAP_Deviation"] = (
        (df["Close"] - df["VWAP"])
        / df["VWAP"]
    )

    # --------------------------------------------------------
    # HFT SCORE
    # --------------------------------------------------------

    score = np.zeros(len(df))

    # Volume spike
    score += np.where(
        df["Volume_Spike"] > 2.0,
        1,
        0
    )

    # Tight spread before move
    score += np.where(
        df["Spread"] < df["Spread_MA"] * 0.8,
        1,
        0
    )

    # Momentum burst
    score += np.where(
        df["Momentum_3"] > 0.03,
        1,
        0
    )

    # ATR expansion
    score += np.where(
        df["ATR"] > df["ATR_MA"] * 1.2,
        1,
        0
    )

    # VWAP breakout
    score += np.where(
        df["VWAP_Deviation"] > 0.02,
        1,
        0
    )

    df["HFT_Score"] = score

    # --------------------------------------------------------
    # SIGNALS
    # --------------------------------------------------------

    df["HFT_Buy"] = (
        df["HFT_Score"] >= HFT_SCORE_THRESHOLD
    )

    return df

# ============================================================
# FETCH DATA
# ============================================================

def fetch_stock(ticker):

    try:

        df = yf.download(
            ticker,
            period=LOOKBACK_PERIOD,
            auto_adjust=True,
            progress=False
        )

        if df.empty or len(df) < 200:
            return None

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df = normalize_index(df)

        df = compute_utbot(df)

        df = compute_hft_score(df)

        return ticker, df

    except:
        return None

# ============================================================
# LOAD ALL DATA
# ============================================================

def load_all_data(tickers):

    all_data = {}

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:

        futures = {
            ex.submit(fetch_stock, t): t
            for t in tickers
        }

        for fut in as_completed(futures):

            result = fut.result()

            if result is not None:

                ticker, df = result

                all_data[ticker] = df

                print("Loaded:", ticker)

    return all_data

# ============================================================
# BACKTEST ENGINE
# ============================================================

def run_backtest(all_data, mode="HFT"):

    cash = INITIAL_CAPITAL

    positions = {}

    trade_log = []

    equity_curve = []

    dates = sorted(
        set(
            d
            for df in all_data.values()
            for d in df.index
        )
    )

    for current_date in dates:

        # ====================================================
        # EXITS
        # ====================================================

        for ticker in list(positions.keys()):

            df = all_data[ticker]

            if current_date not in df.index:
                continue

            row = df.loc[current_date]

            pos = positions[ticker]

            stop_price = (
                pos["entry_price"]
                * (1 - STOP_LOSS_PCT)
            )

            tp_price = (
                pos["entry_price"]
                * (1 + TAKE_PROFIT_PCT)
            )

            exit_trade = False

            if row["Close"] <= stop_price:
                exit_trade = True
                reason = "STOP"

            elif row["Close"] >= tp_price:
                exit_trade = True
                reason = "TARGET"

            elif row["UT_Sell"]:
                exit_trade = True
                reason = "UT_SELL"

            if exit_trade:

                proceeds = (
                    pos["shares"]
                    * row["Close"]
                )

                profit = (
                    proceeds
                    - pos["invested"]
                )

                cash += proceeds

                trade_log.append({

                    "Stock": ticker,

                    "Entry Date": pos["entry_date"],

                    "Exit Date": current_date,

                    "Entry Price": pos["entry_price"],

                    "Exit Price": row["Close"],

                    "Profit": profit,

                    "Return %": (
                        profit / pos["invested"]
                    ) * 100,

                    "Reason": reason
                })

                del positions[ticker]

        # ====================================================
        # ENTRIES
        # ====================================================

        slots = MAX_POSITIONS - len(positions)

        if slots > 0:

            candidates = []

            for ticker, df in all_data.items():

                if ticker in positions:
                    continue

                if current_date not in df.index:
                    continue

                row = df.loc[current_date]

                signal = False

                # ------------------------------------------------
                # HFT ONLY
                # ------------------------------------------------

                if mode == "HFT":

                    signal = bool(row["HFT_Buy"])

                # ------------------------------------------------
                # UT BOT ONLY
                # ------------------------------------------------

                elif mode == "UT":

                    signal = bool(row["UT_Buy"])

                # ------------------------------------------------
                # COMBINED
                # ------------------------------------------------

                elif mode == "COMBINED":

                    signal = (
                        bool(row["HFT_Buy"])
                        and
                        bool(row["UT_Buy"])
                    )

                if signal:

                    score = (
                        row["HFT_Score"]
                        if not pd.isna(row["HFT_Score"])
                        else 0
                    )

                    candidates.append(
                        (ticker, score, row)
                    )

            candidates.sort(
                key=lambda x: x[1],
                reverse=True
            )

            for ticker, score, row in candidates[:slots]:

                allocation = cash / slots

                if allocation <= 0:
                    continue

                shares = allocation / row["Close"]

                cash -= allocation

                positions[ticker] = {

                    "entry_date": current_date,

                    "entry_price": row["Close"],

                    "shares": shares,

                    "invested": allocation
                }

        # ====================================================
        # EQUITY
        # ====================================================

        pv = cash

        for ticker, pos in positions.items():

            df = all_data[ticker]

            if current_date in df.index:

                pv += (
                    pos["shares"]
                    * df.loc[current_date]["Close"]
                )

        equity_curve.append(pv)

    # ========================================================
    # RESULTS
    # ========================================================

    trades = pd.DataFrame(trade_log)

    if trades.empty:

        return {
            "Mode": mode,
            "Final Capital": cash,
            "Win Rate": 0,
            "Sharpe": 0,
            "Max DD": 0,
            "Trades": 0,
            "Equity": equity_curve,
            "TradesDF": trades
        }

    returns = pd.Series(equity_curve).pct_change().dropna()

    sharpe = (
        returns.mean()
        /
        returns.std()
    ) * np.sqrt(252)

    eq = np.array(equity_curve)

    peak = np.maximum.accumulate(eq)

    dd = ((eq - peak) / peak).min()

    win_rate = (
        (trades["Profit"] > 0).mean()
        * 100
    )

    return {

        "Mode": mode,

        "Final Capital": equity_curve[-1],

        "Win Rate": win_rate,

        "Sharpe": sharpe,

        "Max DD": dd * 100,

        "Trades": len(trades),

        "Equity": equity_curve,

        "TradesDF": trades
    }

# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    print("\nLoading NSE tickers...\n")

    tickers = load_nse_tickers()

    print("Total Stocks:", len(tickers))

    all_data = load_all_data(tickers)

    print("\nStocks Loaded:", len(all_data))

    # ========================================================
    # RUN STRATEGIES
    # ========================================================

    hft_result = run_backtest(
        all_data,
        mode="HFT"
    )

    ut_result = run_backtest(
        all_data,
        mode="UT"
    )

    combo_result = run_backtest(
        all_data,
        mode="COMBINED"
    )

    # ========================================================
    # COMPARISON TABLE
    # ========================================================

    comparison = pd.DataFrame([

        {
            "Strategy": hft_result["Mode"],
            "Final Capital": round(hft_result["Final Capital"], 2),
            "Win Rate %": round(hft_result["Win Rate"], 2),
            "Sharpe": round(hft_result["Sharpe"], 2),
            "Max DD %": round(hft_result["Max DD"], 2),
            "Trades": hft_result["Trades"]
        },

        {
            "Strategy": ut_result["Mode"],
            "Final Capital": round(ut_result["Final Capital"], 2),
            "Win Rate %": round(ut_result["Win Rate"], 2),
            "Sharpe": round(ut_result["Sharpe"], 2),
            "Max DD %": round(ut_result["Max DD"], 2),
            "Trades": ut_result["Trades"]
        },

        {
            "Strategy": combo_result["Mode"],
            "Final Capital": round(combo_result["Final Capital"], 2),
            "Win Rate %": round(combo_result["Win Rate"], 2),
            "Sharpe": round(combo_result["Sharpe"], 2),
            "Max DD %": round(combo_result["Max DD"], 2),
            "Trades": combo_result["Trades"]
        }

    ])

    print("\n================ STRATEGY COMPARISON ================\n")

    print(comparison)

    # ========================================================
    # SAVE EXCEL
    # ========================================================

    with pd.ExcelWriter(
        "hft_vs_utbot_results.xlsx",
        engine="openpyxl"
    ) as writer:

        comparison.to_excel(
            writer,
            sheet_name="Comparison",
            index=False
        )

        hft_result["TradesDF"].to_excel(
            writer,
            sheet_name="HFT_Trades",
            index=False
        )

        ut_result["TradesDF"].to_excel(
            writer,
            sheet_name="UT_Trades",
            index=False
        )

        combo_result["TradesDF"].to_excel(
            writer,
            sheet_name="COMBINED_Trades",
            index=False
        )

    print("\nExcel saved: hft_vs_utbot_results.xlsx")

    # ========================================================
    # PLOTS
    # ========================================================

    fig = plt.figure(figsize=(18, 12))

    gs = gridspec.GridSpec(2, 2)

    # --------------------------------------------------------
    # EQUITY CURVES
    # --------------------------------------------------------

    ax1 = fig.add_subplot(gs[0, :])

    ax1.plot(
        hft_result["Equity"],
        label="HFT"
    )

    ax1.plot(
        ut_result["Equity"],
        label="UT BOT"
    )

    ax1.plot(
        combo_result["Equity"],
        label="COMBINED"
    )

    ax1.set_title("Equity Curves")

    ax1.legend()

    ax1.grid(True)

    # --------------------------------------------------------
    # NORMAL DISTRIBUTIONS
    # --------------------------------------------------------

    ax2 = fig.add_subplot(gs[1, 0])

    ax2.hist(
        hft_result["TradesDF"]["Return %"],
        bins=40,
        alpha=0.5,
        density=True,
        label="HFT"
    )

    ax2.hist(
        ut_result["TradesDF"]["Return %"],
        bins=40,
        alpha=0.5,
        density=True,
        label="UT"
    )

    ax2.hist(
        combo_result["TradesDF"]["Return %"],
        bins=40,
        alpha=0.5,
        density=True,
        label="COMBINED"
    )

    ax2.set_title("Return Distribution")

    ax2.legend()

    # --------------------------------------------------------
    # FINAL CAPITAL
    # --------------------------------------------------------

    ax3 = fig.add_subplot(gs[1, 1])

    names = [
        "HFT",
        "UT",
        "COMBINED"
    ]

    vals = [
        hft_result["Final Capital"],
        ut_result["Final Capital"],
        combo_result["Final Capital"]
    ]

    ax3.bar(names, vals)

    ax3.set_title("Final Capital")

    plt.tight_layout()

    plt.savefig(
        "hft_vs_utbot.png",
        dpi=200
    )

    plt.show()

    print("\nChart saved: hft_vs_utbot.png")